# Zurich Airpot data cleaning

In [44]:
# Import necessary libraries
import pandas as pd
import glob
import  matplotlib.pyplot as plt
from pathlib import Path

In [30]:
# Load and concatenate all CSV files from the specified folder
file_paths = glob.glob("./data/zurich*.csv")
df = pd.concat((pd.read_csv(file) for file in file_paths), ignore_index=True)
df

,Date,Type,LocalAirport,ForeignAirport,Airline,Status,Planned,Expected
0,2024-10-28,arrival,Zurich,Geneva,swiss,Landed,21:05,20:58
1,2024-10-28,arrival,Zurich,Berlin,swiss,Landed,21:05,21:03
2,2024-10-28,arrival,Zurich,Paris CDG,swiss,Landed,21:10,21:07
3,2024-10-28,arrival,Zurich,Venice,swiss,Landed,21:15,21:07
4,2024-10-28,arrival,Zurich,Brussels,swiss,Landed,21:20,21:16
...,...,...,...,...,...,...,...,...
8551,2024-11-07,departure,Zurich,Madrid,swiss,Departed,06:50,NaN
8552,2024-11-07,departure,Zurich,Cluj-Napoca,swiss,Departed,06:55,NaN
8553,2024-11-07,departure,Zurich,Vilnius,swiss,Departed,06:55,NaN
8554,2024-11-07,departure,Zurich,Amsterdam,klm,Departed,06:55,NaN


In [31]:
# For better understanding let's rename the column Expected to Actual
df.rename(columns={'Expected': 'Actual'}, inplace=True)

In [32]:
# Fill NaN values in Expected column with values from Planned
df['Actual'].isnull().sum()

np.int64(2167)

In [33]:
df['Actual'] = df['Actual'].fillna(df['Planned']) # these are flights that are simply on time
df['Actual'].isnull().sum()

np.int64(0)

In [34]:
# Convert Planned and Actual columns to datetime format for calculations
df['PlannedDate'] = pd.to_datetime(df['Date'] + " " + df['Planned'])
df['ActualDate'] = pd.to_datetime(df['Date'] + " " + df['Actual'])
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(by='Date', ascending=True).head(5)

,Date,Type,LocalAirport,ForeignAirport,Airline,Status,Planned,Actual,PlannedDate,ActualDate
842,2024-10-21,arrival,Zurich,Copenhagen,swiss,Landed,11:30,11:53,2024-10-21 11:30:00,2024-10-21 11:53:00
876,2024-10-21,arrival,Zurich,Malaga,swiss,Landed,09:10,09:18,2024-10-21 09:10:00,2024-10-21 09:18:00
875,2024-10-21,arrival,Zurich,Kos,edelweiss,Landed,12:45,13:05,2024-10-21 12:45:00,2024-10-21 13:05:00
874,2024-10-21,arrival,Zurich,Heraklion,edelweiss,Landed,12:40,13:03,2024-10-21 12:40:00,2024-10-21 13:03:00
873,2024-10-21,arrival,Zurich,Munich,lufthansa_neu,Landed,12:40,13:07,2024-10-21 12:40:00,2024-10-21 13:07:00


In [35]:
# Filter data for the last two weeks only
two_weeks_ago = pd.to_datetime("today") - pd.Timedelta(weeks=2)
df_last_two_weeks = df[df['Date'] >= two_weeks_ago].copy()
df_last_two_weeks.sort_values(by='Date', ascending=True).head(5)

,Date,Type,LocalAirport,ForeignAirport,Airline,Status,Planned,Actual,PlannedDate,ActualDate
4524,2024-11-03,arrival,Zurich,Shanghai,swiss,Landed,19:00,18:56,2024-11-03 19:00:00,2024-11-03 18:56:00
4908,2024-11-03,departure,Zurich,Seville,edelweiss,Departed,16:55,16:55,2024-11-03 16:55:00,2024-11-03 16:55:00
4907,2024-11-03,departure,Zurich,Florence,swiss,Departed,16:55,16:55,2024-11-03 16:55:00,2024-11-03 16:55:00
4906,2024-11-03,departure,Zurich,Hanover,swiss,Departed,16:50,16:50,2024-11-03 16:50:00,2024-11-03 16:50:00
4905,2024-11-03,departure,Zurich,Valencia,swiss,Departed,16:50,16:45,2024-11-03 16:50:00,2024-11-03 16:45:00


In [39]:
# Add Delay column for further analysis
df_last_two_weeks['Delay'] = ((df_last_two_weeks['ActualDate'] - df_last_two_weeks['PlannedDate'])
                              .dt.total_seconds() / 60)

In [42]:
# Remove unnecessary columns
columns_to_remove = ['Actual', 'Planned', 'Date','Status']

# Drop columns if they exist
df_last_two_weeks = df_last_two_weeks.drop(columns=[col for col in columns_to_remove if col in df_last_two_weeks.columns])
df_last_two_weeks

,Type,LocalAirport,ForeignAirport,Airline,PlannedDate,ActualDate,Delay
4524,arrival,Zurich,Shanghai,swiss,2024-11-03 19:00:00,2024-11-03 18:56:00,-4.0
4525,arrival,Zurich,Warsaw,lot,2024-11-03 19:05:00,2024-11-03 19:00:00,-5.0
4526,arrival,Zurich,Muscat,oman,2024-11-03 19:05:00,2024-11-03 18:57:00,-8.0
4527,arrival,Zurich,Frankfurt,lufthansa_neu,2024-11-03 19:10:00,2024-11-03 19:03:00,-7.0
4528,arrival,Zurich,Lisbon,swiss,2024-11-03 19:15:00,2024-11-03 19:13:00,-2.0
...,...,...,...,...,...,...,...
8551,departure,Zurich,Madrid,swiss,2024-11-07 06:50:00,2024-11-07 06:50:00,0.0
8552,departure,Zurich,Cluj-Napoca,swiss,2024-11-07 06:55:00,2024-11-07 06:55:00,0.0
8553,departure,Zurich,Vilnius,swiss,2024-11-07 06:55:00,2024-11-07 06:55:00,0.0
8554,departure,Zurich,Amsterdam,klm,2024-11-07 06:55:00,2024-11-07 06:55:00,0.0


In [45]:
# Save the final dataset
file_path = Path(f'data/combined_zurich_airport.csv')
# Create the directory if it doesn't exist
file_path.parent.mkdir(parents=True, exist_ok=True)
# Save and overwrite if it already exists
df_last_two_weeks.to_csv(file_path, header=True, index=False)